In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox


# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import generate_single_cell_3d
from render import N_POOLS, render_image, to_rgb
from tape import Tape
from plot import plot_surface_xyz_inline, plot_surface_xyz_html, plot_surface_rgb_html, ortho, ortho_rgb, tau_cmap
from parameter import (P, CellGeometry, CellProfile, CellMarker, Detector,
                       BlobNoise, ClusterNoise, NetworkNoise, FibreNoise, SheetNoise,
                       Optics, TissueGeometry)

%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
SEED = None
N_CAND = 1500
SIZE = 201
TILE = 256
K = 20

# lowest SH degree. 0 = pure size (absorbed by the volume normalisation),
# 1 = shifts the centroid off the seed. 2 = lowest true shape mode.
L_MIN = 2
# 4 is the ceiling of the hardcoded Cartesian forms
L = 4

# microns per LATERAL voxel
UM_PER_VOX = 0.325
# z voxels this many times COARSER than lateral
Z_RATIO = 1.0
# (sz, sy, sx) = voxel SIZE per axis, in lateral units
SPACING = (Z_RATIO, 1.0, 1.0)   
VOL = (128, 128, 128)
IMG = (128, 128, 3)
TISSUE_VOL = (40, 160, 160)

GEOM = CellGeometry()
DETECTOR = Detector()
FRACTIONS = [0.7, 0.3]

# ONE POOL PER (marker, component) PAIR, so two markers using the same noise kind still draw
# independent frozen fields. 3 markers x 3 components = 9.
N_POOLS = 9

# 4) DRAW THE TAPE
tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile=TILE, n_cand=N_CAND, Pool=N_POOLS)
tape.draw3d(vol=VOL, n_cand=N_CAND, Pool=N_POOLS, l_min=L_MIN, L=L)
tape.drawSensor(shape=IMG)

## 1.1) Cell shape gen

In [ ]:
cell = generate_single_cell_3d(tape, GEOM, size=VOL, spacing=SPACING, i=0, L=L, l_min=L_MIN)

In [ ]:
fig = plot_surface_xyz_inline(cell = cell, geom = GEOM)
fig = plot_surface_xyz_html(cell = cell, geom = GEOM,
                        out_path = Path("../renders").resolve())

In [ ]:
lab = cell["cell"].astype(np.int8) + cell["nuc"]        # 0 background, 1 cytoplasm, 2 nucleus
ortho(lab, cmap="viridis", title="mask:  0 background   1 cytoplasm   2 nucleus")
plt.show()

## 1.2) Cell Marker expression

### 1.2.2) Marker espression

In [ ]:
cm, vmin, vmax = tau_cmap(float(cell["tau"][cell["cell"]].max()))
ortho(cell["tau"], cmap=cm, vmin=vmin, vmax=vmax, mask=cell["cell"],
      title=r"$\tau$   (-1 nucleus centre,  0 nuclear envelope,  +1 plasma membrane)")
plt.show()

In [ ]:

PROFILE = CellProfile(
    Geometry = GEOM,
    Markers = dict(

        r = CellMarker(name="r", fluorophore="APC", amp=2.0, polarity=0.2,
            noise_components=[
                ClusterNoise(w=.80, s=1.40, mu= .55, width=.45, sharp=5.0,
                             scale=.35, clust=2.00, fill=.22, soft=.25),
                FibreNoise  (w=.30, s=1.30, mu= .20, width=.70, sharp=4.0,
                             lam=.25, length=6.0),
                NetworkNoise(w=.40, s=1.30, mu= .50, width=.55, sharp=3.5,
                             scale=.80, coherence=.40),
            ]),

        g = CellMarker(name="g", fluorophore="FITC", amp=2.0, polarity=0.2,
            noise_components=[
                BlobNoise   (w=.50, s=1.50, mu= .10, width=.60, sharp=7.5,
                             scale=.45),
                SheetNoise  (w=.60, s=1.50, mu= .40, width=1.20, sharp=6.0,
                             lam=1.90, coherence=.35, length=6.0),
                NetworkNoise(w=.90, s=1.40, mu= .20, width=1.10, sharp=7.0,
                             scale=1.00, coherence=.60),
            ]),

        b = CellMarker(name="b", fluorophore="PE", amp=2.0, polarity=0.2,
            noise_components=[
                ClusterNoise(w=.70, s=1.20, mu=-.55, width=.60, sharp=4.0,
                             scale=.30, clust=1.40, fill=.35, soft=.30),
                BlobNoise   (w=.45, s=1.30, mu=-.60, width=.70, sharp=3.0,
                             scale=.40),
                NetworkNoise(w=.35, s=1.20, mu=-.50, width=.90, sharp=7.0,
                             scale=.70, coherence=.30),
            ]),
    )
)

# edge_softness=0 -> clipped EXACTLY at the plasma membrane. All the softness in the final
# image is made by psf_project and detector, not baked into the object.
img = render_image(tape, PROFILE, cell, spacing=SPACING, um_per_vox=UM_PER_VOX)

rgb = to_rgb(cell["cell"], img)

In [ ]:
ortho_rgb(rgb, title="three markers, orthogonal slices through the cell")
plt.show()

In [ ]:
fig = plot_surface_rgb_html(cell = cell, elong = GEOM.ELONG.v,
                        polar_deg = GEOM.POLAR_DEG.v, azim_deg = GEOM.AZIM_DEG.v, roll_deg = GEOM.ROLL_DEG.v,
                        out_path = Path("../renders").resolve(), volume = rgb)

### 1.2.2) 2d Projection

In [ ]:
from plot import NormalizeData
from optics import kryostat, psf_project, mask_collapse, detector

In [ ]:
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)


subs = {marker: kryostat(v, optics) for marker, v in img.items()}   # keep BOTH vol and z

img_psf = np.stack([psf_project(v, z, optics, PROFILE.Markers[name].fluorophore)
                    for name, (v, z) in subs.items()], -1)

In [ ]:
sub, sub_z = kryostat(cell["cell"], optics)
sub_n, sub_n_z = kryostat(cell["nuc"], optics)


plt.imshow(NormalizeData(img_psf))
plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
# Pre-normalize
img_psf_norm = (img_psf - np.min(img_psf)) / (np.max(img_psf) - np.min(img_psf))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

### 1.2.2) 2d Projection + Detector Model

In [ ]:
markers = list(subs.keys())

img_adu = detector(img_psf, PROFILE, DETECTOR, optics, tape, markers)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(NormalizeData(img_psf), origin="lower")
ax[0].set_title("clean projection (object x PSF)")
ax[1].imshow(NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0)), origin="lower")
ax[1].set_title("detector frame (AF + shot + read)")
ax[2].imshow(NormalizeData(img_adu[..., 0]), cmap="magma", origin="lower")
ax[2].set_title(f"channel 0 ({markers[0]}, {PROFILE.Markers[markers[0]].fluorophore}) in ADU")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Pre-normalize
img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0):
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
    plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0))
print("done")

# 2) Synthetic tissue generation

In [ ]:
### Tissue
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter
from scipy import ndimage as ndi
from render import render_marker

In [ ]:
# For now, just two predesigned cell types
def _marker(name, dye, comps, amp=2.0):
    return CellMarker(name=name, fluorophore=dye, amp=amp, polarity=0.0,
                      noise_components=comps)
PROFILES = {
    "stroma": CellProfile(Color = "red", Markers=dict(
        r=_marker("r", "APC", [ClusterNoise(w=.8, s=1.4, mu=.55, width=.45, sharp=5.,
                                            scale=.35, clust=2.0, fill=.22, soft=.25),
                               FibreNoise(w=.3, s=1.3, mu=.2, width=.7, sharp=4.,
                                          lam=.25, length=6.)], amp=2.4),
        g=_marker("g", "FITC", [BlobNoise(w=.5, s=1.5, mu=.1, width=.6, sharp=7.5, scale=.45),
                                NetworkNoise(w=.9, s=1.4, mu=.2, width=1.1, sharp=7.,
                                             scale=1.0, coherence=.6)], amp=1.2),
        b=_marker("b", "PE", [ClusterNoise(w=.7, s=1.2, mu=-.55, width=.6, sharp=4.,
                                           scale=.30, clust=1.4, fill=.35, soft=.30),
                              BlobNoise(w=.45, s=1.3, mu=-.6, width=.7, sharp=3.,
                                        scale=.4)], amp=2.0))),
    "tumour": CellProfile(Color = "blue", Markers=dict(
        r=_marker("r", "APC", [ClusterNoise(w=.9, s=1.6, mu=.30, width=.8, sharp=4.,
                                            scale=.30, clust=1.2, fill=.45, soft=.20),
                               NetworkNoise(w=.5, s=1.3, mu=.4, width=.7, sharp=4.,
                                            scale=.7, coherence=.4)], amp=0.6),
        g=_marker("g", "FITC", [SheetNoise(w=.7, s=1.5, mu=.4, width=1.2, sharp=6., lam=1.9,
                                           coherence=.35, length=6.),
                                BlobNoise(w=.4, s=1.4, mu=.1, width=.6, sharp=6.,
                                          scale=.5)], amp=2.6),
        b=_marker("b", "PE", [BlobNoise(w=.8, s=1.4, mu=-.65, width=.5, sharp=4., scale=.35),
                              ClusterNoise(w=.6, s=1.3, mu=-.5, width=.7, sharp=4.,
                                           scale=.30, clust=1.0, fill=.5, soft=.25)], amp=2.2))),
}

TISSUE_CELL = CellGeometry(RADIUS=8.0, ROUGH=0.15, ELONG=1.4, NUC_FRAC=0.40, RIM=0.0,
                           NUC_OFFSET=0.8)

FRACTIONS = [0.7, 0.3]
T_POOLS = max(p.n_pools() for p in PROFILES.values())
tape.drawTissue(shape=TISSUE_VOL, n_cand=N_CAND, Pool=T_POOLS)

TG = TissueGeometry()

In [ ]:
from scene import centred_grid_3d, reach_px, _cell_and_nucleus, rotation_matrix
import dataclasses
from dataclasses import dataclass, field

@dataclass
class CellContext:
    """What a rule may look at when deciding one cell's expression."""
    label: int
    # candidate index in the tape
    index: int
    # voxel coords         
    centre: np.ndarray
    geom: CellGeometry
    # labels within NEIGH_RADIUS
    neigh_labels: list
    # label -> type name, for cells already assigned
    types: dict

    def n_neighbours(self):
        return len(self.neigh_labels)

    def neighbour_types(self):
        return [self.types[j] for j in self.neigh_labels if j in self.types]

def stamp_cell(tape, i, geom, grid, sl, centre_world, 
               grow=1.0):
    """Tissue: candidate i stamped at (cy, cx, cz) on a local patch.

    `grow` sizes the cell contour only; `body_grow` scales the rendered cell contour (see
    cell_fields). They are set together (body_grow=grow) to render markers on the packed body.
    """
    return _cell_and_nucleus(
        c_cell=tape["sh"][i], c_nuc=tape["sh2"][i], grid=grid[sl], L=L, l_min=L_MIN,
        nuc_corr=geom.NUC_CORR.v, nuc_frac=geom.NUC_FRAC.v, radius=geom.RADIUS.v,
        rough=geom.ROUGH.v, beta=geom.BETA.v,
        rot=rotation_matrix(geom.POLAR_DEG.v, geom.AZIM_DEG.v, geom.ROLL_DEG.v),
        elong=geom.ELONG.v, rim=geom.RIM.v, nuc_offset=geom.NUC_OFFSET.v,
        off_dir=tape["u_offdir"][i], off_mag=tape["u_offmag"][i],
        grow=grow, centre=centre_world,
        rough_nuc=geom.NUC_ROUGH.v, beta_nuc=geom.NUC_BETA.v)

def vox_to_world(pts_xyz, shape, spacing):
    """(x,y,z) voxel indices -> the world coordinates centred_grid_3d uses."""
    nz, ny, nx = shape
    sz, sy, sx = spacing
    return (np.asarray(pts_xyz, float)
            - np.array([(nx - 1) / 2, (ny - 1) / 2, (nz - 1) / 2])) * np.array([sx, sy, sz])

def patch_grid(centre_xyz, reach, shape):
    """Box of half-width `reach` around a centre, clipped to the volume."""
    nz, ny, nx = shape
    cx, cy, cz = centre_xyz
    return (slice(max(0, int(cz) - reach), min(nz, int(cz) + reach + 1)),
            slice(max(0, int(cy) - reach), min(ny, int(cy) + reach + 1)),
            slice(max(0, int(cx) - reach), min(nx, int(cx) + reach + 1)))

def cell_geometry(base, ttape, i, tg):
    """This cell's own geometry: the base shape plus its frozen size / orientation draws.
    """
    r = float(np.clip(base.RADIUS.v * np.exp(tg.SIZE_SIGMA.v * ttape["z_size"][i]),
                      base.RADIUS.lo, base.RADIUS.hi))
    nf = float(np.clip(base.NUC_FRAC.v * np.exp(tg.NUC_FRAC_SIGMA.v * ttape["z_nucfrac"][i]),
                       base.NUC_FRAC.lo, base.NUC_FRAC.hi))
    o = ttape["u_orient3"][i]
    return dataclasses.replace(base, RADIUS=r, NUC_FRAC=nf, POLAR_DEG=float(180.0 * o[0]),
                               AZIM_DEG=float(360.0 * o[1]), ROLL_DEG=float(360.0 * o[2]))

def thin(pts, order, min_dist):
    keep = np.ones(len(pts), bool)
    for i, j in cKDTree(pts).query_pairs(float(min_dist), output_type="ndarray"):
        keep[i if order[i] > order[j] else j] = False
    return np.flatnonzero(keep)

def support_mask(w, XY_scale_vox=40.0, Z_scale_vox=40.0, cover=0.75):
    """Smooth the frozen field, then threshold at a quantile so `cover` IS the realised fraction."""
    z = gaussian_filter(w.astype(np.float32),
                        (float(Z_scale_vox), float(XY_scale_vox), float(XY_scale_vox)), truncate=3.0)
    z = (z - z.mean()) / (z.std() + 1e-12)
    return z >= np.quantile(z, 1.0 - float(np.clip(cover, 0.0, 1.0)))

def build_tissue(tape, TG, shape, base_geom, spacing):
    sup = support_mask(w = tape["support3"], XY_scale_vox = TG.SUPPORT_SCALE.v, 
                    Z_scale_vox = TG.SUPPORT_SCALE_Z.v,
                    cover = TG.COVER.v)
    keep = thin(tape["xyz"], tape["order"], TG.MIN_DIST.v)
    cx, cy, cz = tape["xyz"][keep].T
    nz, ny, nx = shape
    keep = keep[sup[np.clip(cz.astype(int), 0, nz - 1),
                    np.clip(cy.astype(int), 0, ny - 1),
                    np.clip(cx.astype(int), 0, nx - 1)]]
    
    grid = centred_grid_3d(shape, spacing)
    best = np.full(shape, np.inf, np.float32)
    labels = np.zeros(shape, np.int32)
    nuc_labels = np.zeros(shape, np.int32)
    tau_img = np.zeros(shape, np.float32)
    geoms, slices, centres_w = {}, {}, {}

    for n, i in enumerate(keep, start=1):
        g = cell_geometry(base_geom, tape, i, TG)
        sl = patch_grid(tape["xyz"][i],
                         reach_px(R=g.RADIUS.v, elong=g.ELONG.v, rough=g.ROUGH.v, grow=TG.GROW.v), 
                         shape)
        if any(s.stop - s.start <= 0 for s in sl):
            continue
        cw = vox_to_world(tape["xyz"][i], shape, spacing)
        f = stamp_cell(tape, i, g, grid, sl, cw, grow=1.0)

        # tesselation to decide which cell is closest to each pixel 
        win = (f["d"] < best[sl]) & (f["d"] <= TG.GROW.v) & sup[sl]
        best[sl] = np.where(win, f["d"], best[sl])
        labels[sl] = np.where(win, n, labels[sl])
        tau_img[sl] = np.where(win, f["tau"], tau_img[sl])
        nuc_labels[sl] = np.where(win, f["nuc"] * n, nuc_labels[sl])
        geoms[n], slices[n], centres_w[n] = g, sl, cw
        
    # keep only the piece holding the seed; a two-piece "cell" is a wrong annotation
    orphan = 0
    for n, slc in enumerate(ndi.find_objects(labels), start=1):
        if slc is None:
            continue
        m = labels[slc] == n
        cc, k = ndi.label(m)
        if k > 1:
            main = 1 + int(np.argmax(np.bincount(cc.ravel())[1:]))
            drop = m & (cc != main)
            labels[slc][drop] = 0
            nuc_labels[slc][drop] = 0
            tau_img[slc][drop] = 0.0
            orphan += int(drop.sum())
            orphan += int(drop.sum())
    
    present = sorted(set(np.unique(labels)) - {0})
    cvox = tape["xyz"][keep]
    tree = cKDTree(cvox)
    neigh = {n: [j + 1 for j in tree.query_ball_point(cvox[n - 1], TG.NEIGH_RADIUS.v)
                 if j + 1 != n] for n in present}
    neigh = {n: [j for j in v if j in neigh] for n, v in neigh.items()}
    
    info = dict(labels_present=present, cand_idx={n: int(keep[n - 1]) for n in present},
                geoms=geoms, slices=slices, centres_w=centres_w,
                centres_vox={n: cvox[n - 1] for n in present},
                neighbours=neigh, support=sup, orphan_vox=orphan,
                packing=float((labels > 0).sum() / max(sup.sum(), 1)))
    return labels, nuc_labels, tau_img, info



labels, nuc_labels, tau_img, info = build_tissue(tape=tape, TG=TG, shape=TISSUE_VOL, base_geom=TISSUE_CELL, spacing = SPACING)

In [ ]:

def assign_types(info, tape, Profiles, fractions, rule=None):
    """ecide each cell's type
    """
    type_names = list(Profiles)
    cuts = np.cumsum(np.asarray(fractions, float) / np.sum(fractions))
    types = {}
    for n in info["labels_present"]:
        i = info["cand_idx"][n]
        base = type_names[int(np.searchsorted(cuts, tape["u_type"][i]))]
        ctx = CellContext(label=n, index=i, centre=info["centres_vox"][n],
                          geom=info["geoms"][n], neigh_labels=info["neighbours"][n],
                          types=types)
        types[int(n)] = base if rule is None else rule(ctx, base)
    return types


types = assign_types(info, tape, PROFILES, FRACTIONS)
print(types)

In [ ]:
class TapeDict(dict):
    """dict that also allows attribute access, so it works wherever a Tape does."""
    __getattr__ = dict.__getitem__

# ---------------------------------------------------------------- PHASE 3: appearance
def render_tissue_markers(tape, shape, labels, info, types, profiles, tg,
                          spacing=SPACING, um_per_vox=UM_PER_VOX, verbose=True):
    """
    """
    names = sorted({m for p in profiles.values() for m in p.Markers})
    out = {m: np.zeros(shape, np.float32) for m in names}
    grid = centred_grid_3d(shape, spacing)

    for n in info["labels_present"]:
        prof = profiles[types[n]]
        g, sl = info["geoms"][n], info["slices"][n]
        win = labels[sl] == n
        if not win.any():
            continue
        # the packed body, so a marker fills the claimed territory, not just the free shape
        f = stamp_cell(tape, info["cand_idx"][n], g, grid, sl, info["centres_w"][n],
                          grow=tg.GROW.v)
        ptape = TapeDict(texture_noise=tape["texture_noise"][(slice(None),) + sl],
                         gate_noise=tape["gate_noise"][(slice(None),) + sl])
        off = 0
        for name, marker in prof.Markers.items():
            img = render_marker(tape = ptape, marker = marker, cell = f, spacing = spacing, geom = g, um_per_vox=um_per_vox,
                                edge_softness=0.0, pool_offset=off)
            out[name][sl] = np.where(win, img, out[name][sl])
            off += len(marker.noise_components)
    return out

In [ ]:
t_img = render_tissue_markers(tape, TISSUE_VOL, labels, info, types, PROFILES, TG)
t_rgb = to_rgb(labels > 0, t_img)

In [ ]:
cmap = {name: ct.Color for name, ct in PROFILES.items()}
# PLot tissue mask
mask = info['support'][0].astype(int)
cv = np.array([info["centres_vox"][n] for n in info["labels_present"]])
for i in range(info['support'].shape[0]-1):
    mask += info['support'][i].astype(int)
    plt.scatter(cv[i,0], cv[i,1], c = cmap[types[i+1]])
plt.imshow(mask)

In [ ]:
from plot import NormalizeData
from optics import kryostat, psf_project, mask_collapse, detector

In [ ]:
optics = Optics(um_per_px=UM_PER_VOX, um_per_pz=UM_PER_VOX * Z_RATIO)
tape.drawSensor(shape=(160, 160, 3))

subs = {marker: kryostat(v, optics) for marker, v in t_img.items()}   # keep BOTH vol and z

# We can just use some cell type here as they all use the same channel names
t_psf = np.stack([psf_project(v, z, optics, PROFILES["stroma"].Markers[name].fluorophore)
                  for name, (v, z) in subs.items()], -1)
t_markers = list(subs)
img_adu = detector(t_psf, PROFILES["stroma"], DETECTOR, optics, tape, markers)

In [ ]:
sub, sub_z = kryostat(labels, optics)

# plt.imshow(np.sum(sub, axis=0))
plt.imshow(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[1])

In [ ]:
sub, sub_z = kryostat(labels, optics)
sub_n, sub_n_z = kryostat(nuc_labels, optics)


plt.imshow(NormalizeData(img_adu))
# plt.contour(sub[0], colors="red" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="green" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green
# " , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99, keep_largest=False)[0], colors="yellow" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
# plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
NormalizeData(img_psf).shape

In [ ]:
img_psf_norm = NormalizeData(np.maximum(img_adu - DETECTOR.OFFSET_ADU.v, 0))

def update_image(r_low=0.0, r_high=1.0, r_gamma=1.0,
                 g_low=0.0, g_high=1.0, g_gamma=1.0,
                 b_low=0.0, b_high=1.0, b_gamma=1.0,
                 show_masks=True): # Added missing comma here
    
    adjusted_img = np.zeros_like(img_psf_norm)
    params = [(r_low, r_high, r_gamma), 
              (g_low, g_high, g_gamma), 
              (b_low, b_high, b_gamma)]
    
    for i, (low, high, gamma) in enumerate(params):
        c_data = img_psf_norm[:, :, i]
        denom = max(high - low, 1e-6)
        c_norm = np.clip((c_data - low) / denom, 0.0, 1.0)
        adjusted_img[:, :, i] = c_norm ** gamma

    plt.figure(figsize=(8, 8))
    plt.imshow(adjusted_img)
    
    # Checkbox logic
    if show_masks:
        # plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
        plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(sub, sub_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
        # plt.contour(sub.any(0), colors="red" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.90)[0], colors="blue" , origin="lower", alpha=.55)
        plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.95)[0], colors="green" , origin="lower", alpha=.55)
        # plt.contour(mask_collapse(sub_n, sub_n_z, optics, mask_pct=0.99)[0], colors="yellow" , origin="lower", alpha=.55)
        
    plt.axis('off')
    plt.show()

interact(update_image, 
         r_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         r_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         r_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         # Repeat for G and B...
         g_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         g_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         g_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0),
         b_low=FloatSlider(min=0, max=1, step=0.01, value=0.0),
         b_high=FloatSlider(min=0, max=1, step=0.01, value=1.0),
         b_gamma=FloatSlider(min=0.1, max=5, step=0.1, value=1.0), # Added missing comma here
         show_masks=Checkbox(value=True, description='Show Masks')
)
print("done")